# Step-by-step implementation
The following code demonstrates how to integrate step-back prompting into an RAG pipeline. Let’s break it down step-by-step:
1. Import necessary libraries
2. Set up the OpenAI API key
3. Few-shot learning for step-back prompting
4. Build the step-back prompt
5. Retrieve information
6. Build the RAG chain

## 1. Import necessary libraries

In [1]:
import os
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.runnables import RunnableLambda
from helpers import get_experientiallabs_llm
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper
from langchain_classic import hub

C:\Users\soura\AppData\Local\Temp\ipykernel_4924\2298076215.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import DuckDuckGoSearchAPIWrapper


## 2. Set up the LangSmith

In [2]:
# get_experientiallabs_llm() reads EXPERIENTIALLABS_API_KEY, so no OpenAI key is needed here.

In [3]:
os.environ['LANGSMITH_PROJECT']='Step-Back_Prompting'

## 3. Few-shot learning for step-back prompting

In [4]:
# Few Shot Examples
examples = [
    {
        "input": "Did the Beatles ever write a book?",
        "output": "What types of creative works did the Beatles produce?"
    },
    {
        "input": "Was Albert Einstein a musician?",
        "output": "What fields did Albert Einstein work in?"
    }
]

# We now transform these to example messages
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

## 4. Build the step-back prompt

In [5]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are an expert at world knowledge. Your task is to step back and paraphrase a question to a more generic step-back question, which is easier to answer. Here are a few examples:""",
        ),
        few_shot_prompt,
        ("user", "{question}"),
    ]
)

In [6]:
question_gen = prompt | get_experientiallabs_llm() | StrOutputParser()

In [7]:
question = "Did Leonardo da Vinci invent the printing press?"

In [8]:
question_gen.invoke({"question": question})

'What inventions and innovations is Leonardo da Vinci associated with?'

## 5. Retrieve information

In [9]:
search = DuckDuckGoSearchAPIWrapper(max_results=4)

def retriever(query):
    return search.run(query)

ImportError: Could not import ddgs python package. Please install it with `pip install -U ddgs`.

In [0]:
retriever(question)

In [0]:
retriever(question_gen.invoke({"question": question}))

## 6. Build the RAG chain

In [0]:
response_prompt = hub.pull("langchain-ai/stepback-answer")

In [ ]:
chain = (
    {
        "normal_context": RunnableLambda(lambda x: x["question"]) | retriever,
        "step_back_context": question_gen | retriever,
        "question": lambda x: x["question"],
    }
    | response_prompt
    | get_experientiallabs_llm()
    | StrOutputParser()
)

In [0]:
chain.invoke({"question": question})